# Day 1: Depth-First Search & DLS

In [1]:
# Task 6.1: DFS Find All Paths from 'A' to 'F'
graph_dfs = {
    "A": ["B", "C", "D"],
    "B": ["E", "F"],
    "C": ["F"],
    "D": ["G"],
    "E": [],
    "F": [],
    "G": [],
}


def find_all_paths_dfs(graph, start, end, path=[]):
    path = path + [start]
    if start == end:
        return [path]
    paths = []
    for node in graph.get(start, []):
        if node not in path:
            newpaths = find_all_paths_dfs(graph, node, end, path)
            for p in newpaths:
                paths.append(p)
    return paths


print("--- All Paths from A to F ---")
for p in find_all_paths_dfs(graph_dfs, "A", "F"):
    print(" -> ".join(p))


# Task 6.2: Depth-Limited Search (DLS) Reachability
def dls_reachable_nodes(graph, node, limit, depth=0, visited=None):
    if visited is None:
        visited = set()
    visited.add(node)
    if depth < limit:
        for neighbor in graph.get(node, []):
            dls_reachable_nodes(graph, neighbor, limit, depth + 1, visited)
    return visited


print("\n--- DLS Reachable Nodes ---")
print("Reachable at Limit = 2:", dls_reachable_nodes(graph_dfs, "A", 2))
print("Reachable at Limit = 4:", dls_reachable_nodes(graph_dfs, "A", 4))

--- All Paths from A to F ---
A -> B -> F
A -> C -> F

--- DLS Reachable Nodes ---
Reachable at Limit = 2: {'E', 'D', 'G', 'C', 'A', 'F', 'B'}
Reachable at Limit = 4: {'E', 'D', 'G', 'C', 'A', 'F', 'B'}


# Day 2: Iterative Deepening & Greedy Best-First Search

In [3]:
# Task 7.1: Iterative Deepening Search (IDS) with Evaluation Tracking
def dls_search(graph, node, goal, limit, stats):
    stats["evaluations"] += 1
    if node == goal:
        return True
    if limit <= 0:
        return False
    for neighbor in graph.get(node, []):
        if dls_search(graph, neighbor, goal, limit - 1, stats):
            return True
    return False


def ids_search(graph, start, goal):
    depth = 0
    stats = {"evaluations": 0}
    while True:
        if dls_search(graph, start, goal, depth, stats):
            print(
                f"Goal '{goal}' found at depth {depth}! Total node evaluations: {stats['evaluations']}"
            )
            break
        depth += 1


graph_ids = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": ["G"],
    "F": [],
    "G": [],
}
print("--- IDS Search Execution ---")
ids_search(graph_ids, "A", "G")


# Task 7.2: Greedy Best-First Search (GBFS) with 2 Heuristic Sets
def gbfs(graph, start, goal, heuristic):
    open_list = [start]
    visited = []
    path = []
    while open_list:
        current = min(open_list, key=lambda n: heuristic[n])
        open_list.remove(current)
        path.append(current)
        if current == goal:
            return path
        visited.append(current)
        for neighbor in graph.get(current, []):
            if neighbor not in visited and neighbor not in open_list:
                open_list.append(neighbor)
    return path


heuristic_1 = {"A": 7, "B": 6, "C": 2, "D": 1, "E": 1, "F": 3, "G": 0}
heuristic_2 = {"A": 7, "B": 1, "C": 6, "D": 5, "E": 4, "F": 3, "G": 0}

print("\n--- GBFS Heuristic Impact ---")
print("Path with Heuristic Set 1:", " -> ".join(gbfs(graph_ids, "A", "G", heuristic_1)))
print("Path with Heuristic Set 2:", " -> ".join(gbfs(graph_ids, "A", "G", heuristic_2)))


--- IDS Search Execution ---
Goal 'G' found at depth 3! Total node evaluations: 15

--- GBFS Heuristic Impact ---
Path with Heuristic Set 1: A -> C -> F -> B -> D -> E -> G
Path with Heuristic Set 2: A -> B -> E -> G


# Day 3: Uniform Cost Search

In [4]:
import heapq

# Task 8.1: UCS Step-by-Step Step Updates on Weighted Graph
weighted_graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("D", 2), ("E", 5)],
    "C": [("F", 3)],
    "D": [("G", 1)],
    "E": [("G", 2)],
    "F": [("G", 5)],
    "G": [],
}


def ucs_traced(graph, start, goal):
    pq = [(0, start, [start])]
    visited = set()
    step = 1

    print("--- UCS Priority Queue Trace ---")
    while pq:
        cost, node, path = heapq.heappop(pq)
        print(f"Step {step}: Popped '{node}' | Current Cost g(n) = {cost}")
        step += 1

        if node in visited:
            continue
        visited.add(node)

        if node == goal:
            return cost, path

        for neighbor, edge_cost in graph.get(node, []):
            if neighbor not in visited:
                heapq.heappush(pq, (cost + edge_cost, neighbor, path + [neighbor]))
    return float("inf"), []


cost, path = ucs_traced(weighted_graph, "A", "G")
print(f"\nUCS Optimal Path: {' -> '.join(path)} | Total Cost: {cost}")

--- UCS Priority Queue Trace ---
Step 1: Popped 'A' | Current Cost g(n) = 0
Step 2: Popped 'B' | Current Cost g(n) = 1
Step 3: Popped 'D' | Current Cost g(n) = 3
Step 4: Popped 'C' | Current Cost g(n) = 4
Step 5: Popped 'G' | Current Cost g(n) = 4

UCS Optimal Path: A -> B -> D -> G | Total Cost: 4


# Day 4: A* Search Algorithm (Module 9)

In [5]:
import heapq

# Task 9.1 & 9.2: A* Step Tracing on City Delivery Route
delivery_graph = {
    "Depot": {"Sector_A": 4, "Sector_B": 2},
    "Sector_A": {"Sector_C": 5, "Sector_D": 10},
    "Sector_B": {"Sector_C": 8, "Sector_E": 3},
    "Sector_C": {"Hub": 6},
    "Sector_D": {"Hub": 2},
    "Sector_E": {"Hub": 4},
    "Hub": {},
}

delivery_heuristics = {
    "Depot": 10,
    "Sector_A": 7,
    "Sector_B": 6,
    "Sector_C": 5,
    "Sector_D": 2,
    "Sector_E": 3,
    "Hub": 0,
}


def a_star_traced(graph, start, goal, heuristic):
    open_list = []
    heapq.heappush(open_list, (heuristic[start], 0, start, [start]))
    visited = set()

    print("--- A* Search Execution Trace ---")
    while open_list:
        f, g, current, path = heapq.heappop(open_list)
        print(
            f"Node: {current:<10} | g(n): {g:<2} | h(n): {heuristic[current]:<2} | f(n): {f}"
        )

        if current == goal:
            return path, g

        if current in visited:
            continue
        visited.add(current)

        for neighbor, cost in graph.get(current, {}).items():
            if neighbor not in visited:
                g_new = g + cost
                f_new = g_new + heuristic[neighbor]
                heapq.heappush(
                    open_list, (f_new, g_new, neighbor, path + [neighbor])
                )
    return None, float("inf")


path, cost = a_star_traced(delivery_graph, "Depot", "Hub", delivery_heuristics)
print(f"\nOptimal Route: {' -> '.join(path)} | Total Distance/Cost: {cost}")

--- A* Search Execution Trace ---
Node: Depot      | g(n): 0  | h(n): 10 | f(n): 10
Node: Sector_B   | g(n): 2  | h(n): 6  | f(n): 8
Node: Sector_E   | g(n): 5  | h(n): 3  | f(n): 8
Node: Hub        | g(n): 9  | h(n): 0  | f(n): 9

Optimal Route: Depot -> Sector_B -> Sector_E -> Hub | Total Distance/Cost: 9


# Day 5: Capstone Project — Smart Route Planner
Python

In [6]:
import heapq

# 10-Node City Graph Data Model
city_map = {
    "A": {"B": 3, "C": 5},
    "B": {"D": 4, "E": 7},
    "C": {"F": 2, "G": 8},
    "D": {"H": 3},
    "E": {"H": 2, "I": 5},
    "F": {"I": 4},
    "G": {"J": 3},
    "H": {"J": 1},
    "I": {"J": 2},
    "J": {},
}

heuristics = {
    "A": 9,
    "B": 7,
    "C": 8,
    "D": 4,
    "E": 3,
    "F": 6,
    "G": 3,
    "H": 1,
    "I": 2,
    "J": 0,
}


def bfs_capstone(graph, start, goal):
    queue = [(start, [start])]
    visited = set()
    explored = 0
    while queue:
        curr, path = queue.pop(0)
        explored += 1
        if curr == goal:
            return path, "N/A", explored
        if curr not in visited:
            visited.add(curr)
            for neighbor in graph.get(curr, {}):
                if neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))
    return None, "N/A", explored


def ucs_capstone(graph, start, goal):
    pq = [(0, start, [start])]
    visited = set()
    explored = 0
    while pq:
        cost, curr, path = heapq.heappop(pq)
        explored += 1
        if curr in visited:
            continue
        visited.add(curr)
        if curr == goal:
            return path, cost, explored
        for neighbor, edge_cost in graph.get(curr, {}).items():
            if neighbor not in visited:
                heapq.heappush(pq, (cost + edge_cost, neighbor, path + [neighbor]))
    return None, float("inf"), explored


def astar_capstone(graph, start, goal, h):
    pq = [(h[start], 0, start, [start])]
    visited = set()
    explored = 0
    while pq:
        f, g, curr, path = heapq.heappop(pq)
        explored += 1
        if curr == goal:
            return path, g, explored
        if curr in visited:
            continue
        visited.add(curr)
        for neighbor, cost in graph.get(curr, {}).items():
            if neighbor not in visited:
                heapq.heappush(
                    pq, (g + cost + h[neighbor], g + cost, neighbor, path + [neighbor])
                )
    return None, float("inf"), explored


# Execution & Summary Table
start_node, goal_node = "A", "J"
results = [
    ("BFS", *bfs_capstone(city_map, start_node, goal_node)),
    ("UCS", *ucs_capstone(city_map, start_node, goal_node)),
    ("A*", *astar_capstone(city_map, start_node, goal_node, heuristics)),
]

print("\n=== CAPSTONE: SEARCH ALGORITHM COMPARISON TABLE ===")
print(
    f"{'Algorithm':<12} | {'Path Found':<22} | {'Cost':<6} | {'Nodes Explored':<14}"
)
print("-" * 62)
for name, path, cost, nodes in results:
    path_str = "->".join(path) if path else "N/A"
    print(f"{name:<12} | {path_str:<22} | {str(cost):<6} | {nodes:<14}")

print("\nBest algorithm for shortest path: A*")


=== CAPSTONE: SEARCH ALGORITHM COMPARISON TABLE ===
Algorithm    | Path Found             | Cost   | Nodes Explored
--------------------------------------------------------------
BFS          | A->C->G->J             | N/A    | 12            
UCS          | A->B->D->H->J          | 11     | 9             
A*           | A->B->D->H->J          | 11     | 5             

Best algorithm for shortest path: A*
